In [1]:
import numpy as np
import pandas as pd

In [2]:
'''Tune These As Need Be'''

INITIAL_POPULATION_SIZE = 1_000_000
NUMBER_OF_GENES = 7
POPULATION_MIN = 0
POPULATION_MAX = 10

In [3]:
rng = np.random.default_rng(seed=34)

In [4]:
'''Define Sampling Distribution Here'''

POPULATION = rng.uniform(POPULATION_MIN,POPULATION_MAX,size=(INITIAL_POPULATION_SIZE,NUMBER_OF_GENES))

In [5]:
df = pd.DataFrame(POPULATION,columns=[f"Gene-{i+1}" for i in range(NUMBER_OF_GENES)])
print(df)

          Gene-1    Gene-2    Gene-3    Gene-4    Gene-5    Gene-6    Gene-7
0       0.040282  8.721769  2.427424  6.510613  4.840924  7.885335  8.807226
1       8.997220  5.209929  9.641656  9.113512  6.777658  7.609161  1.787088
2       8.684861  9.995017  9.109405  1.165497  3.876537  9.252288  0.914883
3       3.119389  6.714327  6.274956  2.865582  7.209405  0.081035  8.554845
4       5.828755  0.186167  6.549266  3.972703  5.975747  4.732233  2.904059
...          ...       ...       ...       ...       ...       ...       ...
999995  5.206164  2.116167  1.593342  5.663486  9.521542  3.754951  3.881112
999996  1.950935  3.430173  3.379256  3.741741  9.961430  7.645897  9.084430
999997  5.771389  2.639792  0.971303  0.781373  6.615676  0.363722  4.018719
999998  6.417423  7.329794  2.488871  5.346950  8.446460  7.079568  2.691352
999999  5.091003  0.926987  2.773877  2.490921  0.728791  3.608999  5.601582

[1000000 rows x 7 columns]


In [6]:
'''Define Finction Here to Maximize, Ensure Fitness is allways Positive'''

# Current func = L2 norm of Vectors

def fitness(chromosome):
    return np.mean(chromosome ** 2)

In [7]:
# Finding Best-Fit Chromosome of the Population

Best_Fit = np.argmax(np.apply_along_axis(fitness,1,POPULATION),axis=0)

In [8]:
print("Initial Best Fit\n",df.iloc[Best_Fit])

Initial Best Fit
 Gene-1    8.490557
Gene-2    9.345235
Gene-3    9.272050
Gene-4    9.523020
Gene-5    9.166738
Gene-6    9.715843
Gene-7    9.751878
Name: 830567, dtype: float64


In [9]:
def crossover(chromosome1,chromosome2,alpha=0.7):
    return alpha*chromosome1 + (1-alpha)*np.sqrt(chromosome1*chromosome2)

In [10]:
def mutation(chromosome,*,min_,max_):
    mutator = np.random.random(size=chromosome.shape)
    return np.clip(chromosome+mutator,min_,max_)

In [11]:
PROBA_CROSSOVER = 0.7
PROBA_MUTATION = 0.1

In [12]:
def evolve(Population, *, epochs=100, stagnation_limit=10):
    global fitness,mutation,crossover, PROBA_CROSSOVER, PROBA_MUTATION, INITIAL_POPULATION_SIZE, NUMBER_OF_GENES, POPULATION_MIN, POPULATION_MAX
    epochs_done = 0
    
    best_fitness_so_far = -np.inf
    stagnation = 0
    while (epochs_done < epochs) and (stagnation < stagnation_limit):
        epochs_done += 1
        # Roulette Wheel Selection
        weights = np.apply_along_axis(fitness, 1, Population)

        elite_index = np.argmax(weights)
        elite = Population[elite_index].copy()
        elite_fitness = weights[elite_index]
        
        weights /= np.sum(weights)
        
        parent_indices = np.random.choice(
            len(Population),
            size=len(Population),
            replace=True,
            p=weights
        )    
        parents = Population[parent_indices]

        # CrossOver
        children = []

        assert len(Population) % 2 == 0, "Population size must be even."
        for i in range(0, len(parents), 2):
            p1 = parents[i]
            p2 = parents[i + 1]
            if np.random.random() < PROBA_CROSSOVER:
                c1 = crossover(p1.copy(), p2.copy())
                c2 = crossover(p2.copy(), p1.copy())
            else:
                c1 = p1.copy()
                c2 = p2.copy()

            children.extend([c1, c2])

        children = np.array(children)

        # Mutation
        for i in range(len(children)):
            if rng.random() < PROBA_MUTATION:
                children[i] = mutation(children[i],min_=POPULATION_MIN,max_=POPULATION_MAX)

        # Fitness Calculations
        weights = np.apply_along_axis(fitness, 1, children)

        worst_index = np.argmin(weights)

        children[worst_index] = elite
        
        weights[worst_index] = elite_fitness
        
        best = weights.max()
        
        if best > best_fitness_so_far:
            best_fitness_so_far = best
            stagnation = 0
        else:
            stagnation += 1

        Population = children
        print(f"Epochs Done : {epochs_done}, BEST Fitness : {best}")
    
    if stagnation >= stagnation_limit:
        print("Stopped due to stagnation.")
    else:
        print("Maximum epochs reached.")
    return Population

In [13]:
Evolved_Population = evolve(POPULATION,epochs=100)

Epochs Done : 1, BEST Fitness : 91.9101373266249
Epochs Done : 2, BEST Fitness : 91.9101373266249
Epochs Done : 3, BEST Fitness : 92.756992651819
Epochs Done : 4, BEST Fitness : 92.756992651819
Epochs Done : 5, BEST Fitness : 92.756992651819
Epochs Done : 6, BEST Fitness : 92.756992651819
Epochs Done : 7, BEST Fitness : 93.30941067991823
Epochs Done : 8, BEST Fitness : 96.76792169969745
Epochs Done : 9, BEST Fitness : 96.76792169969745
Epochs Done : 10, BEST Fitness : 98.09095574099469
Epochs Done : 11, BEST Fitness : 98.09095574099469
Epochs Done : 12, BEST Fitness : 98.09095574099469
Epochs Done : 13, BEST Fitness : 98.09095574099469
Epochs Done : 14, BEST Fitness : 98.77840016678785
Epochs Done : 15, BEST Fitness : 100.0
Epochs Done : 16, BEST Fitness : 100.0
Epochs Done : 17, BEST Fitness : 100.0
Epochs Done : 18, BEST Fitness : 100.0
Epochs Done : 19, BEST Fitness : 100.0
Epochs Done : 20, BEST Fitness : 100.0
Epochs Done : 21, BEST Fitness : 100.0
Epochs Done : 22, BEST Fitness :

In [14]:
df = pd.DataFrame(Evolved_Population,columns=[f"Gene-{i+1}" for i in range(NUMBER_OF_GENES)])

In [15]:
print("Discovered Best Fit\n",df.iloc[np.argmax(np.apply_along_axis(fitness,1,Evolved_Population),axis=0)])

Discovered Best Fit
 Gene-1    10.0
Gene-2    10.0
Gene-3    10.0
Gene-4    10.0
Gene-5    10.0
Gene-6    10.0
Gene-7    10.0
Name: 32725, dtype: float64
